# Session 2: 모델 학습

## 학습 목표

1. **Pipeline 개념** 이해: 전처리 → 모델 학습을 하나의 파이프라인으로 묶어 관리
2. **XGBoost** 모델로 대출 승인 여부 예측 모델 학습
3. 학습된 모델을 **파일로 저장**하여 API에서 사용할 수 있도록 준비

### 왜 XGBoost인가?

- **성능**: 정형 데이터(tabular data)에서 최고 수준의 성능을 보입니다.
- **속도**: Gradient Boosting 대비 학습 속도가 빠릅니다.
- **해석 가능성**: Feature importance를 통해 모델의 판단 근거를 설명할 수 있습니다.
- **실무 활용도**: Kaggle 대회, 금융권 실무에서 가장 많이 사용되는 모델 중 하나입니다.

### Pipeline이란?

```
원본 데이터 → [전처리(Scaler)] → [모델(XGBoost)] → 예측 결과
```

Pipeline은 여러 단계를 하나로 묶어 다음과 같은 장점을 제공합니다:
- 학습/예측 시 동일한 전처리가 자동 적용됩니다.
- 코드가 깔끔해지고 실수를 방지합니다.
- 모델 저장 시 전처리 단계까지 함께 저장됩니다.

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')

print('라이브러리 로드 완료!')

라이브러리 로드 완료!


In [41]:
! pip install joblib

In [42]:
# 여기에 코드를 작성하세요
df = pd.read_csv('../data/loan_data.csv')

In [43]:
df.head(2)

,나이,성별,연소득,근속연수,주거형태,신용점수,기존대출건수,연간카드사용액,부채비율,대출신청액,대출목적,상환방식,대출기간,승인여부
0,42,여,4500,16,전세,613,2,3140,28.5,3000,자동차,원금균등,36,1
1,36,남,2400,4,월세,437,3,1260,54.2,2200,주택구입,원리금균등,36,0


In [80]:
df.대출목적.value_counts()

대출목적
6    289
2    265
5    263
1    219
4    215
0    144
3    105
Name: count, dtype: int64

## 범주형 변수 인코딩

머신러닝 모델은 **숫자만** 입력으로 받을 수 있습니다.  
따라서 문자열로 된 범주형 변수(성별, 주거형태 등)를 숫자로 변환해야 합니다.

### LabelEncoder를 사용하는 이유

- 각 범주를 0, 1, 2, ... 등 정수로 변환합니다.
- XGBoost는 트리 기반 모델이라 LabelEncoder의 순서 정보가 문제가 되지 않습니다.
  - (선형 모델이라면 OneHotEncoder가 더 적합합니다)
- 나중에 API에서 새로운 데이터를 예측할 때도 동일한 인코더를 사용해야 하므로 **저장**해둡니다.

In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   나이       1500 non-null   int64  
 1   성별       1500 non-null   object 
 2   연소득      1500 non-null   int64  
 3   근속연수     1500 non-null   int64  
 4   주거형태     1500 non-null   object 
 5   신용점수     1500 non-null   int64  
 6   기존대출건수   1500 non-null   int64  
 7   연간카드사용액  1500 non-null   int64  
 8   부채비율     1500 non-null   float64
 9   대출신청액    1500 non-null   int64  
 10  대출목적     1500 non-null   object 
 11  상환방식     1500 non-null   object 
 12  대출기간     1500 non-null   int64  
 13  승인여부     1500 non-null   int64  
dtypes: float64(1), int64(9), object(4)
memory usage: 164.2+ KB


In [45]:
df['성별']

0       여
1       남
2       남
3       여
4       여
       ..
1495    남
1496    남
1497    남
1498    남
1499    여
Name: 성별, Length: 1500, dtype: object

In [46]:
# 여기에 코드를 작성하세요
df['주거형태']

0       전세
1       월세
2       월세
3       자가
4       월세
        ..
1495    자가
1496    자가
1497    전세
1498    자가
1499    월세
Name: 주거형태, Length: 1500, dtype: object

In [47]:
# label_encoder = LabelEncoder()

In [48]:
# df['성별'] = label_encoder.fit_transform( df['성별'] )

In [49]:
# df['주거형태'] = label_encoder.fit_transform( df['주거형태'] )

In [50]:
# df['대출목적'] = label_encoder.fit_transform( df['대출목적'] )

In [51]:
# df['상환방식'] = label_encoder.fit_transform( df['상환방식'] )

In [52]:
# 위와 같은 방식은, 맨 마지막 인코더만 기억하기 때문에, 예측할때 엉뚱한 결과가 나온다.

In [53]:
cat_cols = ['성별', '주거형태', '대출목적', '상환방식']
label_encoders = {}
for col in cat_cols :
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le    

In [54]:
df

,나이,성별,연소득,근속연수,주거형태,신용점수,기존대출건수,연간카드사용액,부채비율,대출신청액,대출목적,상환방식,대출기간,승인여부
0,42,1,4500,16,2,613,2,3140,28.5,3000,4,1,36,1
1,36,0,2400,4,0,437,3,1260,54.2,2200,6,2,36,0
2,43,0,4900,15,0,623,2,3210,28.2,3800,4,2,48,1
3,51,1,2500,22,1,608,1,1720,35.7,5600,5,2,36,0
4,35,1,3900,3,0,549,0,1630,49.5,7300,2,2,24,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,56,0,2300,10,1,618,1,900,43.5,1100,2,1,36,1
1496,56,0,4300,14,1,573,0,2630,24.8,2800,1,1,36,1
1497,48,0,3400,12,2,597,4,1330,64.1,5600,0,2,60,0
1498,47,0,4400,11,1,554,0,2460,46.5,2500,6,2,36,1


In [55]:
# 숫자로 되어있는 데이터를 피처스케일링 한다. => 학습이 잘 되도록. 
df

,나이,성별,연소득,근속연수,주거형태,신용점수,기존대출건수,연간카드사용액,부채비율,대출신청액,대출목적,상환방식,대출기간,승인여부
0,42,1,4500,16,2,613,2,3140,28.5,3000,4,1,36,1
1,36,0,2400,4,0,437,3,1260,54.2,2200,6,2,36,0
2,43,0,4900,15,0,623,2,3210,28.2,3800,4,2,48,1
3,51,1,2500,22,1,608,1,1720,35.7,5600,5,2,36,0
4,35,1,3900,3,0,549,0,1630,49.5,7300,2,2,24,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,56,0,2300,10,1,618,1,900,43.5,1100,2,1,36,1
1496,56,0,4300,14,1,573,0,2630,24.8,2800,1,1,36,1
1497,48,0,3400,12,2,597,4,1330,64.1,5600,0,2,60,0
1498,47,0,4400,11,1,554,0,2460,46.5,2500,6,2,36,1


In [56]:
y = df['승인여부']

In [57]:
X = df.drop('승인여부', axis= 1)

In [58]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42, stratify= y)

## Pipeline이 왜 중요한가?

### Pipeline 없이 코드를 작성하면:

```python
# 학습 시
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
model.fit(X_train_scaled, y_train)

# 예측 시 (실수하기 쉬움!)
X_new_scaled = scaler.transform(X_new)  # 이걸 빼먹으면 결과가 완전히 달라짐
model.predict(X_new_scaled)
```

### Pipeline을 사용하면:

```python
# 학습 시
pipeline.fit(X_train, y_train)

# 예측 시 (자동으로 스케일링 적용)
pipeline.predict(X_new)  # 전처리 + 예측이 한 번에!
```

### StandardScaler가 필요한 이유

- 연소득(수천만원)과 기존대출건수(한 자리)의 스케일 차이가 큽니다.
- StandardScaler는 평균=0, 표준편차=1로 변환하여 스케일을 통일합니다.
- XGBoost는 트리 기반이라 스케일링이 필수는 아니지만, Pipeline 학습 목적으로 포함합니다.

In [59]:
# 여기에 코드를 작성하세요
pipeline = Pipeline( [ ( 'scaler' , StandardScaler()  ) , 
            ( 'model',  XGBClassifier(n_estimators = 100, 
                                     max_depth = 5, 
                                     learning_rate = 0.1, 
                                     random_state = 42,
                                     eval_metric='logloss')  ) ] )

In [60]:
# 여기에 코드를 작성하세요
pipeline

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='logloss',
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=100, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [61]:
# 학습
pipeline.fit( X_train, y_train )

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='logloss',
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=100, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [62]:
# 평가
y_pred = pipeline.predict(X_test)

In [63]:
X_test.shape

(300, 13)

In [64]:
y_pred

array([1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1,
       1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0,
       1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1,
       1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0,
       0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1,
       0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0,
       1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1,
       0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1,
       1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0,
       0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0])

In [65]:
y_test

282     0
1467    0
1239    1
1036    1
299     1
       ..
1252    1
1377    1
1367    1
172     0
1160    0
Name: 승인여부, Length: 300, dtype: int64

In [66]:
confusion_matrix(y_test, y_pred)

array([[ 94,  23],
       [ 25, 158]], dtype=int64)

In [67]:
accuracy_score(y_test, y_pred)

0.84

In [68]:
xgb_model = pipeline.named_steps['model']

In [69]:
feature_names = X.columns

In [70]:
xgb_model.feature_importances_

array([0.07554234, 0.03992147, 0.16540527, 0.11995751, 0.05343637,
       0.10529502, 0.06502684, 0.04368238, 0.04770951, 0.15581502,
       0.02363884, 0.06785981, 0.03670966], dtype=float32)

In [71]:
pd.Series(xgb_model.feature_importances_ , index= feature_names).sort_values(ascending=False)

연소득        0.165405
대출신청액      0.155815
근속연수       0.119958
신용점수       0.105295
나이         0.075542
상환방식       0.067860
기존대출건수     0.065027
주거형태       0.053436
부채비율       0.047710
연간카드사용액    0.043682
성별         0.039921
대출기간       0.036710
대출목적       0.023639
dtype: float32

In [72]:
os.makedirs('../models' , exist_ok=True)

In [73]:
joblib.dump( pipeline, '../models/loan_pipeline.pkl')

['../models/loan_pipeline.pkl']

In [74]:
joblib.dump( label_encoders, '../models/label_encoders.pkl')

['../models/label_encoders.pkl']

In [75]:
joblib.dump(  list(X.columns)  , '../models/feature_names.pkl' )

['../models/feature_names.pkl']

In [76]:
# 저장된 모델을 불러와서, 신규 데이터를 예측해보자.
# 나이 : 35, 성별 : 남, 연소득 : 5000, 근속연수 : 8, 주거형태 : 자가, 신용점수 : 720, 
# 기존대출건수 : 1, 연간카드사용액 : 2400, 부채비율 : 25.0, 대출신청액 : 3000,
# 대출목적 : 주택구입, 상환방식 : 원리금균등,  대출기간 : 36

In [77]:
X.head(1)

,나이,성별,연소득,근속연수,주거형태,신용점수,기존대출건수,연간카드사용액,부채비율,대출신청액,대출목적,상환방식,대출기간
0,42,1,4500,16,2,613,2,3140,28.5,3000,4,1,36


In [78]:
# 여기에 코드를 작성하세요


In [79]:
# 여기에 코드를 작성하세요


## 정리

### 이번 세션에서 한 것

1. **LabelEncoder**로 범주형 변수(성별, 주거형태, 대출목적, 상환방식)를 숫자로 변환
2. **Pipeline**으로 StandardScaler + XGBClassifier를 하나로 묶어 학습
3. 모델 성능을 Accuracy, Classification Report, Confusion Matrix로 평가
4. **Feature Importance**로 모델의 판단 근거 확인
5. **joblib**으로 모델과 인코더를 파일로 저장
6. 저장된 모델을 로드하여 단일 예측 테스트 성공

### 저장된 파일

| 파일 | 용도 |
|------|------|
| `models/loan_pipeline.pkl` | Pipeline (스케일러 + XGBoost 모델) |
| `models/label_encoders.pkl` | 범주형 변수 인코더 |
| `models/feature_names.pkl` | 피처 이름 목록 |

